In [ ]:
# -*- coding: utf-8 -*-
"""
RD_CA_DENOISING_CLEAN_PDE.ipynb
Lightning Fast, Self-Organizing RD-CA stabilized via Point-wise MLPs, 
Exact Diffusion, and Single-Step Supervision.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import time
import asyncio
import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from collections import deque

torch.backends.cudnn.benchmark = True

# =============================================================================
# Helper: Stable Low-Frequency Initialization
# =============================================================================
def generate_low_freq_noise(batch_size, grid, device):
    scale = max(8, grid // 8)
    noise = torch.rand(batch_size, 3, scale, scale, device=device)
    noise = F.interpolate(noise, size=(grid, grid), mode='bicubic', align_corners=False)
    return torch.clamp(noise, 0.0, 1.0)

# =============================================================================
# Turner BZ CA Ground Truth
# =============================================================================
def bz_step(x, alpha=1.2, beta=1.0, gamma=1.0):
    x_pad = F.pad(x, (1, 1, 1, 1), mode='circular')
    diffused = F.avg_pool2d(x_pad, 3, stride=1, padding=0)
    a, b, c = diffused.chunk(3, dim=1)
    a1 = a + a * (alpha * b - gamma * c)
    b1 = b + b * (beta * c - alpha * a)
    c1 = c + c * (gamma * a - beta * b)
    return torch.cat([a1, b1, c1], dim=1).clamp(0, 1)

# =============================================================================
# Architecture: Ultra-Fast Point-wise PDE Reaction
# =============================================================================
class RD_CA_CleanPDE(nn.Module):
    def __init__(self, in_ch=3, C=32, hidden_mult=2):
        super().__init__()
        self.in_ch = in_ch
        self.C = C
        
        # 1x1 Convs completely drop the compute load while perfectly matching BZ physics.
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, C * hidden_mult, 1),
            nn.GELU(),
            nn.Conv2d(C * hidden_mult, C * hidden_mult, 1),
            nn.GELU(),
            nn.Conv2d(C * hidden_mult, in_ch, 1)
        )
        
        # Initialize projection to zero for stability
        nn.init.normal_(self.net[-1].weight, std=0.01)
        nn.init.zeros_(self.net[-1].bias)

    def get_diffused_local(self, x):
        """
        Exact physical match.
        """
        x_pad = F.pad(x, (1, 1, 1, 1), mode='circular')
        return F.avg_pool2d(x_pad, 3, stride=1, padding=0)

    def get_diffused_fast(self, x):
        return self.get_diffused_local(x)

    def react_step(self, x, x_diff):
        dx = self.net(x_diff)
        return torch.clamp(x_diff + dx, 0.0, 1.0)

# =============================================================================
# State Pool 
# =============================================================================
class StatePool:
    def __init__(self, size=64, grid=128, device='cuda'):
        self.size = size
        self.grid = grid
        self.device = device
        self.x_pool = None
        self.bz_pool = None
        self.age = torch.zeros(size, dtype=torch.long, device=device)

    def sample(self, batch_size):
        if self.x_pool is None:
            noise = generate_low_freq_noise(self.size, self.grid, self.device)
            self.bz_pool = noise.clone()
            self.x_pool = noise.clone()

        idx = np.random.choice(self.size, batch_size, replace=False)
        x_batch = self.x_pool[idx].clone()
        bz_batch = self.bz_pool[idx].clone()

        variance = x_batch.var(dim=[2, 3]).mean(dim=1)
        dead_mask = (variance < 1e-4) | (self.age[idx] > 2000)
        
        if dead_mask.any():
            num_dead = dead_mask.sum().item()
            fresh_noise = generate_low_freq_noise(num_dead, self.grid, self.device)
            bz_batch[dead_mask] = fresh_noise.clone()
            x_batch[dead_mask] = fresh_noise.clone()
            self.age[idx[dead_mask]] = 0

        fresh_noise = generate_low_freq_noise(1, self.grid, self.device)
        bz_batch[0] = fresh_noise.clone()
        x_batch[0] = fresh_noise.clone()
        self.age[idx[0]] = 0

        return x_batch, bz_batch, idx

    def update(self, idx, x_new, bz_new):
        self.x_pool[idx] = x_new.detach()
        self.bz_pool[idx] = bz_new.detach()
        self.age[idx] += 1

# =============================================================================
# AsyncDOR Components
# =============================================================================
class AsyncDOR:
    def __init__(self, shader_func, train_coro_func=None, res=128, fps_limit=30, quality=60):
        self.shader_func = shader_func
        self.train_coro_func = train_coro_func
        self.res = res
        self.fps_limit = fps_limit
        self.quality = quality
        
        self._task = None
        self.train_task = None
        self._running = False
        self._paused = False
        self._pause_offset = 0.0
        self._unpause_wall = None
        self.t = 0.0
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        
        y, x = torch.meshgrid(torch.linspace(-1, 1, res), torch.linspace(-1, 1, res), indexing='ij')
        self.uv = torch.stack((x, y), dim=-1).to(self.device)

        self.img_widget = widgets.Image(format='jpeg', width=res*3, height=res)
        self.fps_label = widgets.Label(value='Ready')

        btn_layout = widgets.Layout(width='32px', height='32px', padding='0px')
        self.btn_toggle = widgets.ToggleButton(value=False, icon='play', button_style='success', layout=btn_layout)
        self.btn_toggle.observe(self._on_toggle, names='value')

        self.btn_pause = widgets.ToggleButton(value=False, icon='pause', button_style='warning', layout=btn_layout)
        self.btn_pause.observe(self._on_pause, names='value')

        self.btn_reset = widgets.Button(icon='refresh', button_style='info', layout=btn_layout)
        self.btn_reset.on_click(lambda _: self.reset_time())

        self.ui = widgets.VBox([
            self.img_widget,
            widgets.HBox([self.btn_toggle, self.btn_pause, self.btn_reset, self.fps_label], layout=widgets.Layout(align_items='center')),
        ])

    def start(self):
        if self._running: return
        self._running = True
        self.btn_toggle.icon = 'stop'
        self.btn_toggle.button_style = 'danger'
        self.btn_toggle.value = True
        self._task = asyncio.ensure_future(self._loop())
        if self.train_coro_func is not None:
            self.train_task = asyncio.ensure_future(self.train_coro_func())

    def stop(self):
        self._running = False
        if self._task and not self._task.done(): self._task.cancel()
        if self.train_task and not self.train_task.done(): self.train_task.cancel()
        self.btn_toggle.icon = 'play'
        self.btn_toggle.button_style = 'success'
        self.btn_toggle.value = False

    def reset_time(self):
        self.t = 0.0
        self._pause_offset = 0.0
        self._unpause_wall = time.time()

    def _on_toggle(self, change):
        if change['new']: self.start()
        else: self.stop()

    def _on_pause(self, change):
        if change['new']:
            self._paused = True
            self._pause_offset = self.t
            self.btn_pause.icon = 'play'
        else:
            self._paused = False
            self._unpause_wall = time.time()
            self.btn_pause.icon = 'pause'

    async def _loop(self):
        dt = 1.0 / self.fps_limit
        encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), self.quality]
        self._unpause_wall = time.time()
        frame_times = deque(maxlen=30)
        frame_count = 0
        min_send_interval = 1.0 / 35.0
        last_send = 0.0

        try:
            while self._running:
                loop_start = time.time()
                if self._paused:
                    await asyncio.sleep(0.05)
                    continue

                self.t = self._pause_offset + (time.time() - self._unpause_wall)
                with torch.no_grad():
                    img_tensor = self.shader_func(self.uv, self.t)
                frame_count += 1

                now = time.time()
                if now - last_send >= min_send_interval:
                    img_np = (img_tensor.clamp(0, 1).mul_(255)).byte().cpu().numpy()
                    if img_np.shape[-1] >= 3: img_np = img_np[:, :, 2::-1]
                    _, enc_data = cv2.imencode('.jpg', img_np, encode_param)
                    self.img_widget.value = enc_data.tobytes()
                    last_send = time.time()

                process_time = time.time() - loop_start
                frame_times.append(process_time)
                if frame_count % 2 == 0:
                    avg = sum(frame_times) / len(frame_times)
                    fps_display = 1.0 / avg if avg > 0 else 0
                    self.fps_label.value = f"FPS: {fps_display:.0f} | t={self.t:.2f}"
                
                wait = dt - process_time
                await asyncio.sleep(max(0.01, wait))
        except asyncio.CancelledError: pass

# =============================================================================
# Side-by-Side Display Management 
# =============================================================================
class DisplayState:
    def __init__(self, model, grid_size):
        self.model = model
        self.grid_size = grid_size
        self.device = next(model.parameters()).device
        self.reset_latent()

    def reset_latent(self):
        with torch.no_grad():
            noise = generate_low_freq_noise(1, self.grid_size, self.device)
            self.display_x_rec = noise.clone()
            self.display_x_cf = noise.clone()
            self.display_bz = noise.clone()

    def shader(self, uv, t):
        with torch.no_grad():
            self.display_bz = bz_step(self.display_bz)
            
            # Left: Local Diffuse
            x_diff_rec = self.model.get_diffused_local(self.display_x_rec)
            self.display_x_rec = self.model.react_step(self.display_x_rec, x_diff_rec)

            # Middle: "Fast" Diffuse (Now symmetrically mapped)
            x_diff_cf = self.model.get_diffused_fast(self.display_x_cf)
            self.display_x_cf = self.model.react_step(self.display_x_cf, x_diff_cf)
            
            return torch.cat([self.display_x_rec[0], self.display_x_cf[0], self.display_bz[0]], dim=2).permute(1, 2, 0)

# =============================================================================
# Live Plotting Logic
# =============================================================================
def get_loss_plot_bytes(loss_history, step):
    fig, ax = plt.subplots(figsize=(5, 3), dpi=90)
    plot_len = min(len(loss_history), 2000)
    y = loss_history[-plot_len:]
    ax.plot(y, color='#1f77b4', alpha=0.3, label="MSE Loss")
    if len(y) > 20:
        smoothed = np.convolve(y, np.ones(20)/20, mode='valid')
        ax.plot(range(19, len(y)), smoothed, color='red', linewidth=2, label="Trend")
    ax.set_yscale('log')
    ax.set_title(f"Step: {step} | Clean PDE Denoising", fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout(pad=1.0)
    fig.canvas.draw()
    img_np = np.asarray(fig.canvas.buffer_rgba())[:, :, :3]
    plt.close(fig)
    _, enc = cv2.imencode('.jpg', cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR), [int(cv2.IMWRITE_JPEG_QUALITY), 85])
    return enc.tobytes()

# =============================================================================
# Setup & Async Training Loop 
# =============================================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
GRID = 128
BATCH = 4
PLOT_EVERY_N_STEPS = 50 

model = RD_CA_CleanPDE(in_ch=3, C=32, hidden_mult=2).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5) 
pool = StatePool(size=64, grid=GRID, device=device)
display_state = DisplayState(model, grid_size=GRID)

class TrainState:
    def __init__(self):
        self.step = 0
        self.loss_history = []
        self.best_loss = float('inf')

train_ctx = TrainState()

async def train_loop():
    t_start = time.time()
    steps_since_start = 0
    try:
        while True:
            train_ctx.step += 1
            steps_since_start += 1
            x_batch, _, idx = pool.sample(BATCH)
            
            # Phase-Space Noise Injection:
            # Prevents rollout-drift without needing slow BPTT unrolling
            x_pred = torch.clamp(x_batch + torch.randn_like(x_batch) * 0.02, 0.0, 1.0)
            
            # Single-step exact target mapping
            with torch.no_grad(): 
                target = bz_step(x_pred)
            
            x_diff = model.get_diffused_local(x_pred)
            x_next = model.react_step(x_pred, x_diff)
            
            # 1x1 convolutions cannot physically bleed spatial information,
            # so exact MSE is drastically faster and perfectly stable.
            loss = F.mse_loss(x_next, target)
            
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            
            pool.update(idx, x_next.detach(), target.detach())
            
            current_loss = loss.item()
            train_ctx.loss_history.append(current_loss)
            
            if current_loss < train_ctx.best_loss:
                train_ctx.best_loss = current_loss
                
            if train_ctx.step % PLOT_EVERY_N_STEPS == 0:
                iter_per_sec = steps_since_start / (time.time() - t_start)
                train_speed_label.value = f"Speed: {iter_per_sec:.2f} it/s | Best Loss: {train_ctx.best_loss:.5f}"
                t_start = time.time()
                steps_since_start = 0
                loss_widget.value = get_loss_plot_bytes(train_ctx.loss_history, train_ctx.step)
            await asyncio.sleep(0)
    except asyncio.CancelledError: 
        pass 

# =============================================================================
# UI Display
# =============================================================================
dor = AsyncDOR(display_state.shader, train_coro_func=train_loop, res=GRID, fps_limit=30)
dor.img_widget.width = str(GRID * 3) 
header = widgets.HTML(value="<b>[ Left: Fast Local ] | [ Middle: Fast Local ] | [ Right: Ground Truth BZ ]</b>")
loss_widget = widgets.Image(format='jpeg', width=450, height=270)
train_speed_label = widgets.Label(value='Speed: - it/s | Best Loss: -')

ui_layout = widgets.VBox([
    header,
    widgets.HBox([dor.ui, loss_widget], layout=widgets.Layout(align_items='center')),
    train_speed_label
])
display(ui_layout)

original_reset = dor.reset_time
def decoupled_reset():
    original_reset()
    display_state.reset_latent()
dor.reset_time = decoupled_reset
dor.start()